# Code printer: AMReX

Goal: print a `NumericalSystemModel` as an AMReX C++ header.

Prerequisite: [NumericalSystemModel](numerical_system_model.ipynb).

In [1]:
from zoomy_core.model.models import SME, Newtonian, NavierSlip, StressFree
from zoomy_core.numerics import NumericalSystemModel, ReconstructionSpec
from zoomy_core.systemmodel import SystemModel
from zoomy_core.systemmodel.operations import symbolic_spectrum
from zoomy_core.transformation.to_amrex import AmrexModel

sm = SystemModel.from_model(
    SME(level=0, closures=[Newtonian(), NavierSlip(), StressFree()])).apply(symbolic_spectrum())
nsm = NumericalSystemModel.from_system_model(
    sm, reconstruction=ReconstructionSpec(order=1), depth_regularization="aux")
code = AmrexModel(nsm).create_code()
print(len(code.splitlines()), "lines")

424 lines


In [2]:
import re
import textwrap

meta = [l.strip() for l in code.splitlines() if re.search(r"n_dof_q |n_dof_qaux|dimension ", l)]
flux = textwrap.dedent(re.search(r" *AMREX_GPU_HOST_DEVICE[^\n]*\n\s*static [^\n]* flux\(.*?\n    \}", code, re.S).group(0))
print("\n".join(meta))
print()
print(flux)

static constexpr int n_dof_q    = 3;
static constexpr int n_dof_qaux = 4;
static constexpr int dimension  = 1;

AMREX_GPU_HOST_DEVICE AMREX_FORCE_INLINE
static amrex::SmallMatrix<amrex::Real,3,1> flux(
    const amrex::Real* Q,
    const amrex::Real* Qaux,
    const amrex::Real* p) noexcept
{
amrex::SmallMatrix<amrex::Real,3,1> res;
res(0, 0) = 0;
res(1, 0) = Q(2, 0);
res(2, 0) = Qaux(3, 0)*amrex::Math::powi<2>(Q(2, 0));
return res;
}


In [3]:
assert code.lstrip().startswith("#pragma once") and "struct Model" in code
assert "n_dof_q    = 3" in code and "dimension  = 1" in code
assert "res(1, 0) = Q(2, 0);" in flux and "amrex::Math::powi<2>(Q(2, 0))" in flux